<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [6]</a>'.</span>

In [1]:
# Parameters
UF = "BRASIL"


In [2]:
# Parameters
UF = "AC"


## Seção 3.2. - Equipamentos utilizados

Este notebook reproduz as tabelas 7 e 8, e a figura 24 da seção 3.2 do Relatório Anual de Qualidade do Ar

LEIA ATENTAMENTE ANTES DE EXECUTAR ESTE  SCRIPT

1. ATUALIZAÇÃO DOS CAMINHOS:

   Este código precisa saber em qual pasta do seu computador o projeto está salvo.
   Como cada computador possui uma estrutura de pastas diferente, você deve 
   atualizar a variável de caminho bruto (quando indicado) para
   apontar para o diretório correto na sua máquina antes de rodar o script.


2. PRESERVAÇÃO DA ESTRUTURA DE PASTAS:

   Este código funciona de forma integrada com as demais pastas, scripts e
   arquivos auxiliares exatamente na ORGANIZAÇÃO fornecida no projeto.

   -> Não mova arquivos ou pastas de lugar.

   -> Não altere o nome dos diretórios ou arquivos.

   Caso a estrutura fornecida seja alterada, as importações e chamadas de dados 
   irão falhar e o código não irá funcionar.


### Tabela 7 — Número de estações por categoria de monitoramento

Tabela estática que exibe, para cada Unidade Federativa (UF), a contagem de estações de monitoramento separadas por **categoria**: Indicativa, Referência e Não declarado.  

As UFs são agrupadas por **região geográfica** e cada linha exibe a bandeira do estado. A linha BR totaliza os valores nacionais.  

Zeros são substituídos por `-` para facilitar a leitura.

A função `table05()` executa todo o processamento e a estilização da tabela.

In [3]:
import pandas as pd
aqmData = pd.read_csv('https://arquivos.lcqar.ufsc.br/data/databases/stations/Monitoramento_QAr_BR.csv')
mapping_df = pd.read_csv('https://arquivos.lcqar.ufsc.br/rqar-national-guide-files/data/dicionarios/CODIGO_POLUENTES_HTML.csv')
path_bandeiras = 'https://arquivos.lcqar.ufsc.br/rqar-national-guide-files/_static/bandeiras/'


In [4]:
# Este codigo depende de flagTables.py (função table05()). que lê `/Monitoramento_QAr_BR.csv e as imagens de bandeiras em _static/bandeiras/{UF}.png.

# Importações necessárias
import importlib
import flagTables as flagtab

importlib.reload(flagtab)
html_tabela7 = flagtab.table05(aqmData.copy(), path_bandeiras, uf=UF)

,UF,Indicativa,Referência,Não declarado
,,,,
,Norte,,,
,AC,30,-,-
,,,,


In [5]:
import webbrowser
import os

output_dir = "outputs"
output_path = os.path.join(output_dir, "tabela7.html")

os.makedirs(output_dir, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_tabela7)

webbrowser.open(output_path)

True

### Figura 24 — Distribuição espacial das estações por categoria

Mapa interativo do Brasil mostrando a localização geográfica de cada estação de monitoramento colorida conforme a **categoria do equipamento utilizado**:  
laranja para **Indicativa**, verde para **Referência** e cinza para **Não declarado**. O mapa permite filtrar a visualização das categorias.

Ao passar o mouse sobre cada ponto, são exibidos: ID OEMA, status, poluentes monitorados, número de poluentes medidos e categoria. 

Ao clicar no ponto são exibidos: UF, ID_OEMA, latitude, longitude, status, categoria, poluente e numero de poluentes medidos.


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [6]:
# Este codigo depende de scripts/spatialFigures.py(função spatial_rede_monitoramento_interative()), que requer Monitoramento_QAr_BR.csv com as colunas de latitude e longitude preenchidas.

# Importações necessárias
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import scripts.spatialFigures as spaFig
import importlib; importlib.reload(spaFig)
from matplotlib.colors import ListedColormap

# Cores por categoria: Indicativa = laranja, Não declarado = cinza, Referência = verde
cmap = ListedColormap(["orange", "gray", "green"])

# Tooltips exibidos ao passar o mouse sobre cada estação no mapa
tooltips = ["ID_OEMA", "STATUS", "POLUENTE", "N° Poluentes Medidos", "CATEGORIA"]

m = spaFig.spatial_rede_monitoramento_interative(aqmData.copy(), "CATEGORIA", tooltips, cmap, uf=UF)
m

TypeError: spatial_rede_monitoramento_interative() got multiple values for argument 'uf'

In [ ]:
import webbrowser

output_dir = "outputs"
output_path = os.path.join(output_dir, "figura24.html")

os.makedirs(output_dir, exist_ok=True)

m.save(output_path)

webbrowser.open(output_path)


###Tabela 8 — Pontos de monitoramento por poluente

Tabela interativa e filtrável que lista cada ponto de monitoramento individualmente, exibindo as colunas: UF (com bandeira), ID OEMA, poluentes medidos, status de operação, categoria e forma de funcionamento. Permite realizar o download dos dados filtrados através da ferramenta interativa.

O painel lateral de **SearchPanes** permite filtrar simultaneamente por UF, status e categoria.  
Linhas com UF não declarada, vazia ou nula são removidas antes da exibição.

A tabela é construída por meio das funções `flagTable()` e `tabela_iterativa()` do módulo `flagTables`.

In [ ]:
# Importações necessárias

import pandas as pd
import numpy as np
import importlib
import warnings
import flagTables as flagtab

warnings.filterwarnings('ignore')
importlib.reload(flagtab)

# Criação da tabela via flagTables

columnsSelector = ["", "UF", "ID_OEMA", "Poluente", "Status", "Categoria", "Funcionamento"]
aqmDisplay = flagtab.flagTable(aqmData.copy(), path_bandeiras, columnsSelector, uf=UF)

searchPaneColumns = [1, 4, 5]

# Remove linhas com UF não declarada / vazia / nula
uf_series = aqmDisplay['UF'].astype(str).str.strip()
mask_invalid_uf = (
    uf_series.eq('') |
    uf_series.str.lower().isin(['não declarado', 'nao declarado', 'nan']) |
    aqmDisplay['UF'].isna()
)
aqmDisplay = aqmDisplay[~mask_invalid_uf].copy()

# Exibe tabela interativa limpa
# Garante que o índice não seja usado como coluna
aqmDisplay = aqmDisplay.reset_index(drop=True)

html_tabela8 = flagtab.tabela_iterativa(aqmDisplay, searchPaneColumns)

In [ ]:
# Salva a tabela interativa como um arquivo HTML e abre no navegador
import webbrowser
import os

output_dir = "outputs"
output_path = os.path.join(output_dir, "tabela8.html")

os.makedirs(output_dir, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_tabela8)

webbrowser.open(output_path)